# Day 1 (final) — frozen pipeline, verification, save

**What this is.** Not a rerun of Day 1's discovery. Every open question from last week is settled;
this notebook freezes the answers into `ct_utils.py`, re-verifies them at the final settings, and
saves the config that Day 3 loads.

**What changed since the last run.**

| | Last week | Now | Why |
|---|---|---|---|
| `max_feature_nodes` | 16384 | 32768 | **32768** Corpus now allows 20-token prompts; a 15-token prompt already reached 13,081 nodes |
| `MAX_TOKENS` | 15 | **20** | MCQ, multi-hop and entity pairs could not be written properly in 14 content tokens |
| Metrics | defined inline, copied between notebooks | **one module, `ct_utils.py`** | Copy-paste drift between Day 1 and Day 3 is a silent bug waiting to happen |
| `error_mass_final_pos` | a metric | **a diagnostic** | Structurally zero once the cap isn't binding; it was measuring cap saturation |
| Position summaries | not implemented | **thirds, argmax, Gini, second-last** | These are what the pre-registration (6.3) specified |
| Saturation | checked by hand | **graph rejected by an exception** | Saturation silently fakes a category effect |

**Budget.** New week, 30 GPU hours. This notebook uses roughly half an hour.

Accelerator: **GPU T4 x2**\

**Deviation:**
**Feature cap raised to 32768.** At 24576 the densest pilot prompt (induction, nonce bigrams) had 1,700 features of headroom. Nonce text fragments into many subword tokens and is the most feature-dense input in the corpus. Attribution time was measured to be independent of the cap (13.3s vs 13.4s at 16384 vs 24576), so raising it costs nothing and prevents induction prompts being rejected by the saturation guard.

The embedded ct_utils.py predates the logit-layout fix. Add a line at the top of that notebook saying so and pointing at the current ct_utils.py, or a reader will run it and get the old metrics.

## Part 0 — Install, then restart

Run alone. Then **Run → Restart kernel**, then continue from Part 1.

In [ ]:
!pip install -q circuit-tracer
print("installed — Run > Restart kernel, then continue from Part 1")

## Part 1 — Environment

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, gc, json
import numpy as np, pandas as pd, torch

cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
assert cap >= (7, 0), f"{GPU_NAME} is sm_{cap[0]}{cap[1]}; P100 is unusable. Use T4 x2."
print(f"GPU: {GPU_NAME}  sm_{cap[0]}{cap[1]}")

def hw(peak=False):
    ram = os.popen("free -g | awk 'NR==2{print $7}'").read().strip() or "?"
    disk = os.popen("df -BG /kaggle/working | awk 'NR==2{print $4}'").read().strip()
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    s = f"RAM free {ram} GB | GPU {a:.1f}/{t:.1f} GB | disk free {disk}"
    if peak:
        s += f" | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB"
    print(s)

def free():
    gc.collect(); torch.cuda.empty_cache()

hw()

In [ ]:
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
from huggingface_hub import login, whoami
login(token=_tok)
print("logged in as:", whoami()["name"])
os.makedirs("/kaggle/working/out", exist_ok=True)

from importlib.metadata import version, PackageNotFoundError
def ver(d):
    try: return version(d)
    except PackageNotFoundError: return "unknown"

VERSIONS = {"torch": torch.__version__, "transformers": ver("transformers"),
            "transformer_lens": ver("transformer-lens"), "circuit_tracer": ver("circuit-tracer"),
            "nnsight": ver("nnsight"), "numpy": ver("numpy"), "scipy": ver("scipy"),
            "gpu": GPU_NAME}
for k, v in VERSIONS.items():
    print(f"{k:18s} {v}")

## Part 2 — Load

In [ ]:
from circuit_tracer import ReplacementModel, attribute
import circuit_tracer.graph as ctg

t0 = time.time()
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", backend="nnsight",
    dtype=torch.bfloat16, device="cuda", lazy_encoder=True,
)
print(f"loaded in {time.time()-t0:.0f}s")
free(); hw()

## Part 3 — Frozen configuration

Every value here has a provenance. The comments are the methods section in draft.

In [ ]:
CONFIG = dict(
    config_version="day1-final",
    model="google/gemma-2-2b",
    transcoders="gemma",                 # GemmaScope per-layer, mwhanna/gemma-scope-transcoders
    backend="nnsight",                   # TransformerLens load exceeded ~13 GB system RAM
    dtype="bfloat16",                    # fp32 needs ~14.3 GB; fp16 overflows under Gemma-2 soft-capping
    device="cuda",
    lazy_encoder=True,                   # encoders were the resident half; decoders lazy by default
    n_layers=model.cfg.n_layers,
    split="errors_first",                # matched the library's replacement score (0.7196)
    reshape="layer_major",               # mod-n_tok zero pattern: 26/26 in two buckets, flat under mod 26
    attr_kwargs=dict(
        max_feature_nodes=24576,         # 8192 saturated above ~10 tokens and faked final-position error
        batch_size=32,                   # 64 peaked at 14.3/14.56 GB; 32 peaked at 6.5 GB, no slower
    ),
    max_tokens=20,
    tolerance=1e-4,                      # conservation tolerance under bf16
    versions=VERSIONS,
)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "versions"}, indent=2))

## Part 4 — Write the shared module

One source of truth for the metric pipeline. Day 3 imports this file rather than carrying its own
copy. **Download it and commit it to the repo** — Kaggle sessions do not persist.

In [ ]:
# %%writefile /kaggle/working/ct_utils.py
"""ct_utils - frozen metric pipeline for attribution-graph-triage.

Every decision below was settled on Day 1 and is documented in day1_config.json.
Day 3 imports this module; do not edit it without adding a deviations entry.

Settled:
  - Node types recovered from graph STRUCTURE, not index conventions: error and
    embedding nodes have zero in-degree; logits have zero out-degree.
  - Errors and embeddings sit in ONE contiguous zero-in-degree run, errors first
    (CHOICE = errors_first, confirmed against the library's own replacement score).
  - The 156-style error block is LAYER-MAJOR (confirmed by the mod-n_tok zero
    pattern: 26/26 zeros in two residue buckets, flat noise under mod 26).
  - Error at the BOS and final positions is structurally zero once the feature cap
    is not binding. Position summaries are therefore computed on interior positions.
  - A graph that saturates max_feature_nodes is REJECTED, not recorded: saturation
    injects spurious error mass and would fake a category effect.
"""
import numpy as np
import scipy.sparse as sp
import torch


def to_np(x):
    if torch.is_tensor(x):
        return x.detach().cpu().float().numpy()
    return np.asarray(x)


def contiguous_blocks(idx):
    if len(idx) == 0:
        return []
    out, s, p = [], idx[0], idx[0]
    for i in idx[1:]:
        if i != p + 1:
            out.append((s, p))
            s = i
        p = i
    out.append((s, p))
    return out


def find_nodes(A0, n_tok, n_layers, n_log):
    """Return (err_idx, emb_idx, logit_idx). A0 is indexed [target, source]."""
    nz = np.abs(A0) > 0
    zin = np.where(nz.sum(axis=1) == 0)[0]
    zout = np.where(nz.sum(axis=0) == 0)[0]

    run = max(contiguous_blocks(zin), key=lambda b: b[1] - b[0])
    span = run[1] - run[0] + 1
    want = n_layers * n_tok + n_tok
    if span != want:
        raise ValueError(f"zero-in-degree run is {span}, expected {want}")

    ne = n_layers * n_tok
    err = np.arange(run[0], run[0] + ne)
    emb = np.arange(run[0] + ne, run[1] + 1)

    lg = np.setdiff1d(zout, zin)
    if len(lg) != n_log:
        lg = lg[np.argsort(-np.abs(A0[lg]).sum(axis=1))[:n_log]]
    return err, emb, lg


def influence(A_raw, logit_idx, logit_probs, absorb_idx, max_iter=500):
    """Probability-weighted total path strength from each node to the logits.

    Row-normalise incoming edge magnitudes, then push mass backwards from the
    logits one edge per step. The graph is a DAG, so the series terminates
    exactly rather than converging asymptotically. Returns the normalised
    matrix, the influence vector, mass absorbed at `absorb_idx` per path length,
    and the number of steps taken.
    """
    A = np.abs(A_raw)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    AT = sp.csr_matrix(A.T)

    v = np.zeros(A.shape[0])
    v[logit_idx] = logit_probs
    infl = np.zeros(A.shape[0])
    by_len = []
    for k in range(1, max_iter + 1):
        v = AT @ v
        infl += v
        by_len.append(v[absorb_idx].sum())
        if v.sum() < 1e-12:
            break
    return A, infl, np.array(by_len), k


def gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    if x.sum() <= 0 or len(x) < 2:
        return 0.0
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))


def thirds(v):
    tot = v.sum()
    if tot <= 0:
        return [0.0, 0.0, 0.0]
    return [float(p.sum() / tot) for p in np.array_split(v, 3)]


def pruning_curve(graph, prune_fn, thresholds=(0.95, 0.9, 0.8, 0.7)):
    out = {}
    if prune_fn is None:
        return out
    for t in thresholds:
        try:
            res = prune_fn(graph, node_threshold=t, edge_threshold=t)
            mask = getattr(res, "node_mask", None)
            out[str(t)] = int(mask.sum()) if mask is not None else None
        except Exception as ex:
            out[str(t)] = f"ERR {type(ex).__name__}"
    return out


def graph_metrics(gr, cfg, prune_fn=None):
    """SUPERSEDED (22-09-2026): locates logits structurally and can mis-order them.
    Kept only as a record of the Day 1 implementation. Use graph_metrics_fast."""
    A0 = to_np(gr.adjacency_matrix).astype(np.float64)
    nt = len(to_np(gr.input_tokens))
    nlg = len(to_np(gr.logit_probabilities))
    nl = cfg["n_layers"]
    cap = cfg["attr_kwargs"]["max_feature_nodes"]

    n_feat = A0.shape[0] - (nl * nt + nt + nlg)
    if n_feat >= cap:
        raise ValueError(f"feature cap saturated: {n_feat} >= {cap}")

    err, emb, lg = find_nodes(A0, nt, nl, nlg)

    lw = to_np(gr.logit_probabilities)
    lw = lw / lw.sum()
    A, inf, bl, steps = influence(A0, lg, lw, emb)
    if steps >= 500:
        raise ValueError("influence series did not terminate: orientation is wrong")

    rep = float(inf[emb].sum())
    etot = float(inf[err].sum())
    if abs(rep + etot - 1.0) > cfg["tolerance"]:
        raise ValueError(f"conservation violated: {rep + etot}")

    mask = np.zeros(A.shape[0], bool)
    mask[err] = True
    w = inf * (A.sum(axis=1) > 0)
    comp = float(1 - (w @ A[:, mask].sum(axis=1)) / w.sum())

    grid = inf[err].reshape(nl, nt)          # layer-major, settled Day 1
    by_pos = grid.sum(axis=0)
    by_layer = grid.sum(axis=1)
    interior = by_pos[1:-1] if nt > 2 else by_pos

    return {
        "replacement_score": rep,
        "completeness_score": comp,
        "error_mass_total": etot,
        "mean_path_length": float((np.arange(1, len(bl) + 1) @ bl) / bl.sum()),
        # position summaries on interior positions (BOS and final are structurally zero)
        "err_pos_thirds": thirds(interior),
        "err_pos_argmax_norm": float(np.argmax(interior) / max(len(interior) - 1, 1)),
        "err_pos_gini": gini(interior),
        "err_second_last_pos": float(by_pos[-2]) if nt > 1 else 0.0,
        # layer summaries - always 26 values, comparable across the whole corpus
        "err_layer_thirds": thirds(by_layer),
        "err_layer_argmax": int(np.argmax(by_layer)),
        "err_late_layers": float(by_layer[-6:].sum()),
        # structural-zero diagnostics; should be ~0 on every graph
        "err_bos_pos": float(by_pos[0]),
        "err_final_pos": float(by_pos[-1]),
        # raw profiles, for anything not pre-specified
        "error_by_position": by_pos.tolist(),
        "error_by_layer": by_layer.tolist(),
        "n_nodes": int(A.shape[0]),
        "n_features": int(n_feat),
        "n_edges": int((A0 != 0).sum()),
        "n_tokens": nt,
        "series_steps": int(steps),
        "pruning_nodes": pruning_curve(gr, prune_fn),
    }


# ---------------------------------------------------------------------------
# FAST PATH (added after Day 1 final showed 50-260 s per graph at cap 24576)
#
# graph_metrics above converts the adjacency to dense float64 NumPy on the CPU,
# copying an ~18,000 x 18,000 matrix several times, and calls prune_graph four
# times. graph_metrics_fast does the same computation on the GPU in float32,
# never copies the full matrix off the device, and replaces the library pruning
# curve with a node-influence threshold curve computed from the influence vector
# it already has.
#
# It MUST be validated against graph_metrics on the same graph before use:
# see validate_fast() below.
# ---------------------------------------------------------------------------

def node_count_curve(infl, thresholds=(0.95, 0.9, 0.8, 0.7)):
    """Nodes needed to reach each cumulative share of total influence.

    This is the node-threshold step of the library's prune_graph, without its
    edge pruning or iterative orphan removal. A cheap proxy, reported as such.
    """
    s = np.sort(np.asarray(infl))[::-1]
    tot = s.sum()
    if tot <= 0:
        return {str(t): 0 for t in thresholds}
    c = np.cumsum(s) / tot
    return {str(t): int(np.searchsorted(c, t) + 1) for t in thresholds}


@torch.no_grad()
def graph_metrics_fast(gr, cfg, device="cuda"):
    """GPU float32 metrics, matching circuit_tracer.graph.compute_graph_scores.

    Node layout is taken from the graph, exactly as the library does:
        [features | errors (n_tok * n_layers, layer-major) | tokens | logits]
    with logits the LAST n_logits nodes, in the same order as logit_probabilities.

    An earlier version located logits structurally (zero out-degree). That set also
    contains dead features, and narrowing it by incoming weight REORDERED the logits,
    so each probability was assigned to the wrong node. That was the source of the
    ~0.008 gap from the library. The structural findings are kept as assertions.
    """
    A = gr.adjacency_matrix.to(device=device, dtype=torch.float32, copy=True).abs_()
    N = A.shape[0]
    nt = int(len(to_np(gr.input_tokens)))
    nlg = int(len(to_np(gr.logit_probabilities)))
    nl = cfg["n_layers"]
    cap = cfg["attr_kwargs"]["max_feature_nodes"]

    n_feat = int(len(gr.selected_features))
    if n_feat >= cap:
        raise ValueError(f"feature cap saturated: {n_feat} >= {cap}")
    err_end = n_feat + nl * nt
    tok_end = err_end + nt
    if tok_end + nlg != N:
        raise ValueError(f"layout mismatch: {n_feat}+{nl*nt}+{nt}+{nlg} != {N} nodes")
    err = np.arange(n_feat, err_end)
    emb = np.arange(err_end, tok_end)
    lg = np.arange(N - nlg, N)

    # Structural checks: errors and tokens have no inputs; logits have no outputs.
    nz = A > 0
    in_deg = nz.sum(1).cpu().numpy()
    out_deg = nz.sum(0).cpu().numpy()
    del nz
    if in_deg[n_feat:tok_end].any():
        raise ValueError("an error or token node has incoming edges: layout is wrong")
    if out_deg[lg].any():
        raise ValueError("a logit node has outgoing edges: layout is wrong")

    rs = A.sum(1, keepdim=True)
    A.div_(rs.clamp_min(1e-30))               # rows with no inputs stay zero

    lw = torch.zeros(N, device=device)
    lw[N - nlg:] = torch.as_tensor(to_np(gr.logit_probabilities),
                                   device=device, dtype=torch.float32)
    lw = lw / lw.sum()        # scale cancels in both scores; normalising keeps conservation = 1

    v = lw.clone()
    infl = torch.zeros(N, device=device)
    emb_t = torch.as_tensor(emb, device=device)
    err_t = torch.as_tensor(err, device=device)
    by_len = []
    steps = 0
    for steps in range(1, 501):
        v = v @ A                             # v_new[s] = sum_t v[t] * A[t, s]
        infl += v
        by_len.append(float(v[emb_t].sum()))
        if float(v.sum()) < 1e-12:
            break
    if steps >= 500:
        raise ValueError("influence series did not terminate")

    tok_inf = float(infl[emb_t].sum())
    err_inf = float(infl[err_t].sum())
    if abs(tok_inf + err_inf - 1.0) > cfg["tolerance"]:
        raise ValueError(f"conservation violated: {tok_inf + err_inf}")
    rep = tok_inf / (tok_inf + err_inf)

    # Library completeness: over ALL nodes, weighted by influence PLUS the logit weights.
    non_err = 1 - A[:, err_t].sum(1)
    out_inf = infl + lw
    comp = float((non_err * out_inf).sum() / out_inf.sum())

    inf_np = infl.cpu().numpy()
    del A
    bl = np.array(by_len)
    grid = inf_np[err].reshape(nl, nt)       # layer-major, settled Day 1
    by_pos, by_layer = grid.sum(axis=0), grid.sum(axis=1)
    interior = by_pos[1:-1] if nt > 2 else by_pos
    ni = len(interior)

    return {
        "replacement_score": rep,
        "completeness_score": comp,
        "error_mass_total": err_inf,
        "mean_path_length": float((np.arange(1, len(bl) + 1) @ bl) / bl.sum()),
        "err_pos_thirds": thirds(interior),
        "err_pos_argmax_norm": float(np.argmax(interior) / max(ni - 1, 1)),
        "err_pos_gini": gini(interior),
        "err_pos_gini_norm": gini(interior) * ni / (ni - 1) if ni > 1 else 0.0,
        "err_second_last_pos": float(by_pos[-2]) if nt > 1 else 0.0,
        "err_layer_thirds": thirds(by_layer),
        "err_layer_argmax": int(np.argmax(by_layer)),
        "err_late_layers": float(by_layer[-6:].sum()),
        "err_bos_pos": float(by_pos[0]),
        "err_final_pos": float(by_pos[-1]),
        "error_by_position": by_pos.tolist(),
        "error_by_layer": by_layer.tolist(),
        "n_nodes": int(N),
        "n_features": n_feat,
        "n_interior": ni,
        "n_edges": int(in_deg.sum()),
        "n_tokens": nt,
        "series_steps": int(steps),
        "node_count_curve": node_count_curve(inf_np),
    }


def validate_against_library(gr, cfg, lib_fn, tol=1e-4):
    """The fast path must reproduce circuit_tracer's compute_graph_scores."""
    lib_rep, lib_comp = lib_fn(gr)
    m = graph_metrics_fast(gr, cfg)
    d_rep = abs(m["replacement_score"] - lib_rep)
    d_comp = abs(m["completeness_score"] - lib_comp)
    print(f"replacement   ours {m['replacement_score']:.6f}  library {lib_rep:.6f}  delta {d_rep:.2e}")
    print(f"completeness  ours {m['completeness_score']:.6f}  library {lib_comp:.6f}  delta {d_comp:.2e}")
    assert d_rep < tol and d_comp < tol, "fast path does not match the library"
    print(f"matches the library within {max(d_rep, d_comp):.2e}")
    return m


def validate_fast(gr, cfg, tol=1e-3):
    """Run both implementations on one graph; they must agree before the fast
    path is trusted for the corpus."""
    slow = graph_metrics(gr, cfg, prune_fn=None)
    fast = graph_metrics_fast(gr, cfg)
    keys = ["replacement_score", "completeness_score", "error_mass_total",
            "mean_path_length", "err_pos_gini", "err_late_layers"]
    worst = 0.0
    for k in keys:
        d = abs(slow[k] - fast[k])
        worst = max(worst, d)
        print(f"{k:22s} slow {slow[k]:.6f}  fast {fast[k]:.6f}  delta {d:.2e}")
    for k in ("n_nodes", "n_features", "n_tokens"):
        assert slow[k] == fast[k], f"{k} differs"
    assert worst < tol, f"implementations disagree by {worst:.2e}"
    print(f"\nagree within {worst:.2e}  (tolerance {tol:.0e})")
    return slow, fast


In [ ]:
sys.path.insert(0, "/kaggle/working")
import importlib, ct_utils
importlib.reload(ct_utils)
from ct_utils import graph_metrics, to_np
print("ct_utils loaded:", [n for n in dir(ct_utils) if not n.startswith("_")])

## Part 5 — Verify against the library

The library's own metrics are the reference. Last week: library 0.7196 / 0.9268, ours 0.7271 /
0.9356. The gap is ~150x the noise floor, so it is a genuine methodological difference rather than
rounding — most likely logit-node weighting or normalisation. This cell reads the library source so
the difference can be stated rather than guessed at.

In [ ]:
import inspect
METRIC_FN = None
for n in dir(ctg):
    o = getattr(ctg, n)
    if not n.startswith("_") and callable(o) and o.__doc__ \
       and "eplacement" in o.__doc__ and "ompleteness" in o.__doc__:
        METRIC_FN = o
        break
print("library metric function:", METRIC_FN.__name__ if METRIC_FN else None)
if METRIC_FN is not None:
    print(inspect.getsource(METRIC_FN))

In [ ]:
PROMPT = "The capital of France is"
AK = CONFIG["attr_kwargs"]

g = attribute(prompt=PROMPT, model=model, **AK)          # warm-up
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); g = attribute(prompt=PROMPT, model=model, **AK); t_ref = time.time() - t0

m = graph_metrics(g, CONFIG, prune_fn=ctg.prune_graph)
lib = METRIC_FN(g) if METRIC_FN else (None, None)

print(f"attribution {t_ref:.1f}s")
print(f"replacement   ours {m['replacement_score']:.4f}   library {lib[0]:.4f}   "
      f"delta {abs(m['replacement_score'] - lib[0]):.4f}")
print(f"completeness  ours {m['completeness_score']:.4f}   library {lib[1]:.4f}   "
      f"delta {abs(m['completeness_score'] - lib[1]):.4f}")
print(f"err at BOS {m['err_bos_pos']:.5f}   err at final {m['err_final_pos']:.5f}   "
      f"(both should be ~0)")
CONFIG["library_scores_reference"] = [float(lib[0]), float(lib[1])]
hw(peak=True)

In [ ]:
#NEW
for p in ["The capital of France is", "48+35=",
          "snerrik yolvantz snerrik yolvantz snerrik",
          "Fact: the capital of the state containing Dallas is"]:
    gp = attribute(prompt=p, model=model, **AK)
    mm = graph_metrics_fast(gp, CONFIG)
    print(f"{p[:34]:36s} BOS {mm['err_bos_pos']:.6f}   final {mm['err_final_pos']:.6f}   "
          f"interior max {max(mm['error_by_position'][1:-1]):.6f}")
    del gp; free()

## Part 6 — Saturation at the new ceiling

The one thing that genuinely needs re-checking. A 15-token prompt reached 13,081 nodes at cap
16384; a 20-token prompt could plausibly reach 16,000+. Saturation silently injects error mass, so
verify the longest prompts you will actually run.

In [ ]:
LONG = [
    "After the long meeting finally ended everyone went home to rest and then talked about the",
    "snerrik yolvantz snerrik yolvantz snerrik yolvantz snerrik",
    "Frida Kahlo was born in the year",
    "Sun is a: (a) planet (b) star. Answer: (",
]
rows = []
for p in LONG:
    nt = len(model.tokenizer(p)["input_ids"])
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        gp = attribute(prompt=p, model=model, **AK)
        mm = graph_metrics(gp, CONFIG, prune_fn=ctg.prune_graph)
        rows.append(dict(n_tok=nt, secs=round(time.time()-t0, 1),
                         features=mm["n_features"], cap=AK["max_feature_nodes"],
                         headroom=AK["max_feature_nodes"] - mm["n_features"],
                         err_final=round(mm["err_final_pos"], 5),
                         peak_gb=round(torch.cuda.max_memory_allocated()/1e9, 1),
                         prompt=p[:40]))
        del gp
    except Exception as ex:
        rows.append(dict(n_tok=nt, secs=round(time.time()-t0, 1), features=None, cap=None,
                         headroom=None, err_final=None, peak_gb=None,
                         prompt=f"FAILED {type(ex).__name__}: {str(ex)[:60]}"))
    free()

sat = pd.DataFrame(rows)
print(sat.to_string(index=False))
print("\nWant: headroom comfortably positive, err_final ~0, peak well under 14.56 GB.")
print("graph_metrics raises on saturation, so a FAILED row with 'saturated' means raise the cap.")

#### Testing code for timing (requested by claude)

In [ ]:
p = "After the long meeting finally ended everyone went home to rest and then talked about the"
t = {}
t0 = time.time(); gp = attribute(prompt=p, model=model, **AK); t["attribution"] = time.time() - t0
t0 = time.time(); mm = graph_metrics(gp, CONFIG, prune_fn=None); t["metrics"] = time.time() - t0
t0 = time.time(); _ = ct_utils.pruning_curve(gp, ctg.prune_graph); t["pruning x4"] = time.time() - t0
for k, v in t.items(): print(f"{k:12s} {v:6.1f}s")

# is the cap itself costing time?
q = "Frida Kahlo was born in the year"
for cap_ in (16384, 24576):
    t0 = time.time(); attribute(prompt=q, model=model, max_feature_nodes=cap_, batch_size=32)
    print(f"attribution at cap {cap_}: {time.time()-t0:.1f}s")
free()

In [ ]:
import ct_utils
print("graph_metrics_fast" in dir(ct_utils), ct_utils.__file__)

In [ ]:
gp = attribute(prompt="After the long meeting finally ended everyone went home to rest and then talked about the",
               model=model, **AK)
slow, fast = validate_fast(gp, CONFIG)
t0 = time.time(); graph_metrics_fast(gp, CONFIG); print(f"fast metrics: {time.time()-t0:.1f}s")

In [ ]:
validate_fast()

In [ ]:
CONFIG["metrics_impl"] = "graph_metrics_fast"
CONFIG["fast_vs_slow_max_delta"] = 3.07e-08
CONFIG["pruning"] = "node_count_curve; library prune_graph on a 30-graph subsample"

json.dump(CONFIG, open("/kaggle/working/out/day1_config.json", "w"), indent=2, default=str)
shutil.copy("/kaggle/working/ct_utils.py", "/kaggle/working/out/ct_utils.py")
print(sorted(os.listdir("/kaggle/working/out")))

## Part 7 — Noise floor

In [ ]:
g2 = attribute(prompt=PROMPT, model=model, **AK)
m2 = graph_metrics(g2, CONFIG, prune_fn=ctg.prune_graph)
NOISE = {k: abs(m[k] - m2[k]) for k in ("replacement_score", "completeness_score", "mean_path_length")}
for k, d in NOISE.items():
    print(f"{k:22s} delta {d:.2e}")
print(f"n_nodes identical: {m['n_nodes'] == m2['n_nodes']}   (structure must be exactly reproducible)")
assert m["n_nodes"] == m2["n_nodes"], "graph structure is non-deterministic"
CONFIG["noise_floor"] = NOISE
del g2; free()

## Part 8 — Timing at final settings

In [ ]:
import traceback

PILOT = [
    ("factual",    "The capital of France is"),
    ("factual",    "Fact: the capital of France is"),
    ("arithmetic", "48+35="),
    ("syntax",     "Despite everything that had happened earlier that morning, the keys to the cabinet"),
    ("multi_hop",  "Fact: the capital of the state containing Dallas is"),
    ("code",       "def add(a, b):\n    return"),
    ("induction",  "snerrik yolvantz snerrik yolvantz snerrik"),   # feature-densest category
    ("induction",  "blorp trilve blorp trilve blorp"),
    ("mcq",        "Sun is a: (a) planet (b) star. Answer: ("),
    ("entity",     "Frida Batkin was born in the year"),
    ("obfuscated", "Th3y d3cid3d 70 le4v3 7h3"),
    ("obfuscated", "They dECIDeD to LeAVE ThE"),
]

cap_ = AK["max_feature_nodes"]
rows = []
for i, (cat, p) in enumerate(PILOT):
    torch.cuda.reset_peak_memory_stats()
    try:
        t0 = time.time()
        gp = attribute(prompt=p, model=model, **AK)
        t_attr = time.time() - t0
        t0 = time.time()
        mm = graph_metrics_fast(gp, CONFIG)
        t_met = time.time() - t0

        if i in (0, len(PILOT) - 1):          # re-validate on shortest-ish and a corrupted one
            validate_fast(gp, CONFIG)

        rows.append(dict(cat=cat, n_tok=mm["n_tokens"],
                         attr_s=round(t_attr, 1), metric_s=round(t_met, 2),
                         total_s=round(t_attr + t_met, 1),
                         features=mm["n_features"], headroom=cap_ - mm["n_features"],
                         rep=round(mm["replacement_score"], 3),
                         comp=round(mm["completeness_score"], 3),
                         pos_gini=round(mm["err_pos_gini"], 2),
                         err_bos=round(mm["err_bos_pos"], 6),
                         err_final=round(mm["err_final_pos"], 6),
                         peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 1),
                         prompt=p[:32]))
        del gp
    except Exception:
        print(f"\nFAILED [{cat}] {p!r}")
        traceback.print_exc()
        rows.append(dict(cat=cat, prompt=p[:32]))
    free()

pilot = pd.DataFrame(rows)
print(pilot.to_string(index=False))

In [ ]:
## Summary
ok = pilot.dropna(subset=["rep"])
per = ok.total_s.mean()

print(f"\n{len(ok)}/{len(pilot)} succeeded")
print(f"per graph: mean {per:.1f}s | max {ok.total_s.max():.1f}s | "
      f"metrics share {ok.metric_s.sum() / ok.total_s.sum():.1%}")
print(f"280 graphs: {280*per/3600:.1f} h, x1.5 margin {1.5*280*per/3600:.1f} h   (week budget 30 h)")
print(f"min feature headroom: {ok.headroom.min()}  on [{ok.loc[ok.headroom.idxmin(), 'cat']}]")
print(f"peak GPU: {ok.peak_gb.max():.1f} GB of 14.56")

nz = ok[(ok.err_bos.abs() > 1e-6) | (ok.err_final.abs() > 1e-6)]
print(f"graphs with non-zero BOS/final error: {len(nz)}   (must be 0 for the interior-position rule)")
assert len(nz) == 0, nz[["cat", "prompt", "err_bos", "err_final"]]

print(f"\nreplacement range: {ok.rep.min():.3f} - {ok.rep.max():.3f}   "
      f"(last week's pilot: 0.684 - 0.755)")

CONFIG["seconds_per_graph"] = float(per)
CONFIG["min_feature_headroom_pilot"] = int(ok.headroom.min())

## Part 9 — Save everything, then download

In [ ]:
json.dump(CONFIG, open("/kaggle/working/out/day1_config.json", "w"), indent=2, default=str)
json.dump(m, open("/kaggle/working/out/reference_france.json", "w"), indent=2)
pilot.to_csv("/kaggle/working/out/day1_pilot.csv", index=False)
sat.to_csv("/kaggle/working/out/day1_saturation.csv", index=False)

import shutil
shutil.copy("/kaggle/working/ct_utils.py", "/kaggle/working/out/ct_utils.py")

# prove it round-trips before you close the session
cfg_back = json.load(open("/kaggle/working/out/day1_config.json"))
assert cfg_back["attr_kwargs"]["max_feature_nodes"] == 24576
print(sorted(os.listdir("/kaggle/working/out")))
print("\n*** DOWNLOAD /kaggle/working/out/ NOW. Also: Save Version on this notebook. ***")

In [ ]:
import shutil

# ---- final settings -------------------------------------------------------
CONFIG["attr_kwargs"]["max_feature_nodes"] = 32768     # induction headroom was 1700 at 24576; cap costs no time
AK = CONFIG["attr_kwargs"]
CONFIG.update(
    metrics_impl="graph_metrics_fast",
    fast_vs_slow_max_delta=3.45e-08,
    pruning="node_count_curve; library prune_graph on a 30-graph subsample",
    seconds_per_graph=float(per),
    peak_gpu_gb_pilot=float(ok.peak_gb.max()),
    min_feature_headroom_pilot_at_24576=int(ok.headroom.min()),
)

# ---- reference graph, computed with the implementation Day 3 will use ------
g_ref = attribute(prompt="The capital of France is", model=model, **AK)
ref = graph_metrics_fast(g_ref, CONFIG)
del g_ref; free()

# ---- save ------------------------------------------------------------------
OUT = "/kaggle/working/out"
json.dump(CONFIG, open(f"{OUT}/day1_config.json", "w"), indent=2, default=str)
json.dump(ref, open(f"{OUT}/reference_france.json", "w"), indent=2)
pilot.to_csv(f"{OUT}/day1_pilot.csv", index=False)
sat.to_csv(f"{OUT}/day1_saturation.csv", index=False)
shutil.copy("/kaggle/working/ct_utils.py", f"{OUT}/ct_utils.py")

# ---- prove it round-trips before closing the session -----------------------
back = json.load(open(f"{OUT}/day1_config.json"))
assert back["attr_kwargs"]["max_feature_nodes"] == CONFIG["attr_kwargs"]["max_feature_nodes"]
assert back["metrics_impl"] == "graph_metrics_fast"
assert "graph_metrics_fast" in open(f"{OUT}/ct_utils.py").read(), "saved ct_utils is the OLD version"
assert abs(ref["err_bos_pos"]) < 1e-6 and abs(ref["err_final_pos"]) < 1e-6

print(sorted(os.listdir(OUT)))
print(f"\ncap {AK['max_feature_nodes']} | {per:.1f}s/graph | reference replacement "
      f"{ref['replacement_score']:.4f}")
print("\n*** DOWNLOAD /kaggle/working/out/ NOW, then Save Version ***")

## Done when

- [ ] `ct_utils` agrees with the library's metrics (and you know why they differ by ~0.008)
- [ ] BOS and final-position error ~0 on the reference graph
- [ ] Every long prompt has positive feature headroom at 24576, and `err_final` ~0
- [ ] `n_nodes` exactly reproducible; noise floor recorded
- [ ] Per-graph time recorded
- [ ] Five files downloaded: `day1_config.json`, `reference_france.json`, `day1_pilot.csv`,
      `day1_saturation.csv`, `ct_utils.py`

**Deviation to log:** position summaries are computed on *interior* positions, excluding BOS and
the final token, because error there is structurally zero once the cap does not bind. Including
them would inflate the Gini coefficient for reasons unrelated to the prompt.\
**Feature cap raised to 32768.** At 24576 the densest pilot prompt (induction, nonce bigrams) had 1,700 features of headroom. Nonce text fragments into many subword tokens and is the most feature-dense input in the corpus. Attribution time was measured to be independent of the cap (13.3s vs 13.4s at 16384 vs 24576), so raising it costs nothing and prevents induction prompts being rejected by the saturation guard.